In [0]:

DROP TABLE IF EXISTS adventure_works.bronze.stg_purchase_order_detail;


In [0]:

CREATE TABLE adventure_works.bronze.stg_purchase_order_detail
AS
SELECT *
FROM read_files(
    '/Volumes/adventure_works/default/raw_data/PurchaseOrderDetail.csv',
    format => 'csv',
    header => false,
    inferSchema => false
);


In [0]:

SELECT *
FROM adventure_works.bronze.stg_purchase_order_detail
LIMIT 10;


In [0]:

SELECT
    COUNT(*) AS total_rows
FROM adventure_works.bronze.stg_purchase_order_detail;


In [0]:

DESCRIBE adventure_works.bronze.stg_purchase_order_detail;


In [0]:

SELECT _c0
FROM adventure_works.bronze.stg_purchase_order_detail
LIMIT 5;


In [0]:
DROP TABLE IF EXISTS adventure_works.bronze.stg_purchase_order_detail;

##Creating Bronze Layer

In [0]:

CREATE TABLE adventure_works.bronze.stg_purchase_order_detail
AS
SELECT *
FROM read_files(
    '/Volumes/adventure_works/default/raw_data/PurchaseOrderDetail.csv',
    format => 'csv',
    header => false,
    inferSchema => false,
    delimiter => '\t'
);


In [0]:
DESCRIBE adventure_works.bronze.stg_purchase_order_detail;

In [0]:
SELECT *
FROM adventure_works.bronze.stg_purchase_order_detail
LIMIT 5;

In [0]:

CREATE TABLE adventure_works.bronze.stg_purchase_order_header
AS
SELECT *
FROM read_files(
    '/Volumes/adventure_works/default/raw_data/PurchaseOrderHeader.csv',
    format => 'csv',
    header => false,
    inferSchema => false,
    delimiter => '\t'
);


In [0]:

CREATE TABLE adventure_works.bronze.dim_vendor
AS
SELECT *
FROM read_files(
    '/Volumes/adventure_works/default/raw_data/Vendor.csv',
    format => 'csv',
    header => false,
    inferSchema => false,
    delimiter => '\t'
);


In [0]:

SHOW TABLES IN adventure_works.bronze;


In [0]:

SELECT 'purchase_order_detail' AS table_name,
       COUNT(*) AS total_rows
FROM adventure_works.bronze.stg_purchase_order_detail

UNION ALL

SELECT 'purchase_order_header',
       COUNT(*)
FROM adventure_works.bronze.stg_purchase_order_header

UNION ALL

SELECT 'vendor',
       COUNT(*)
FROM adventure_works.bronze.dim_vendor;


In [0]:

CREATE TABLE adventure_works.bronze.stg_sales_order_header
AS
SELECT *
FROM read_files(
    '/Volumes/adventure_works/default/raw_data/SalesOrderHeader.csv',
    format => 'csv',
    header => false,
    inferSchema => false,
    delimiter => '\t'
);


In [0]:

SELECT COUNT(*) AS total_rows
FROM adventure_works.bronze.stg_sales_order_header;


In [0]:

DESCRIBE adventure_works.bronze.stg_sales_order_header;


##Creating Silver Layer

In [0]:

CREATE SCHEMA IF NOT EXISTS adventure_works.silver;


In [0]:
DESCRIBE adventure_works.bronze.stg_purchase_order_detail;

In [0]:
DESCRIBE adventure_works.bronze.stg_purchase_order_header;

In [0]:
DESCRIBE adventure_works.bronze.dim_vendor;

In [0]:

CREATE TABLE adventure_works.silver.stg_purchase_order_detail
AS
SELECT
    TRY_CAST(_c0 AS INT) AS PurchaseOrderID,
    TRY_CAST(_c1 AS INT) AS PurchaseOrderDetailID,
    TRY_CAST(_c2 AS TIMESTAMP) AS DueDate,
    TRY_CAST(_c3 AS SMALLINT) AS OrderQty,
    TRY_CAST(_c4 AS INT) AS ProductID,
    TRY_CAST(_c5 AS DECIMAL(19,4)) AS UnitPrice,
    TRY_CAST(_c6 AS DECIMAL(19,4)) AS LineTotal,
    TRY_CAST(_c7 AS DECIMAL(8,2)) AS ReceivedQty,
    TRY_CAST(_c8 AS DECIMAL(8,2)) AS RejectedQty,
    TRY_CAST(_c9 AS DECIMAL(9,2)) AS StockedQty,
    TRY_CAST(_c10 AS TIMESTAMP) AS ModifiedDate

FROM adventure_works.bronze.stg_purchase_order_detail

WHERE
    TRY_CAST(_c4 AS INT) IS NOT NULL
    AND TRY_CAST(_c6 AS DECIMAL(19,4)) >= 0;


In [0]:
SELECT COUNT(*) AS total_clean_rows
FROM adventure_works.silver.stg_purchase_order_detail;

In [0]:
DESCRIBE adventure_works.silver.stg_purchase_order_detail;

In [0]:

SELECT
    COUNT(*) AS bronze_total_rows,

    SUM(CASE
        WHEN TRY_CAST(_c4 AS INT) IS NULL
        THEN 1 ELSE 0
    END) AS null_or_invalid_product_id,

    SUM(CASE
        WHEN TRY_CAST(_c6 AS DECIMAL(19,4)) < 0
        THEN 1 ELSE 0
    END) AS negative_line_total

FROM adventure_works.bronze.stg_purchase_order_detail;


In [0]:

CREATE TABLE adventure_works.silver.stg_purchase_order_header
AS
SELECT
    TRY_CAST(_c0 AS INT) AS PurchaseOrderID,
    TRY_CAST(_c1 AS TINYINT) AS RevisionNumber,
    TRY_CAST(_c2 AS TINYINT) AS Status,
    TRY_CAST(_c3 AS INT) AS EmployeeID,
    TRY_CAST(_c4 AS INT) AS VendorID,
    TRY_CAST(_c5 AS INT) AS ShipMethodID,
    TRY_CAST(_c6 AS TIMESTAMP) AS OrderDate,
    TRY_CAST(_c7 AS TIMESTAMP) AS ShipDate,
    TRY_CAST(_c8 AS DECIMAL(19,4)) AS SubTotal,
    TRY_CAST(_c9 AS DECIMAL(19,4)) AS TaxAmt,
    TRY_CAST(_c10 AS DECIMAL(19,4)) AS Freight,
    TRY_CAST(_c11 AS DECIMAL(19,4)) AS TotalDue,
    TRY_CAST(_c12 AS TIMESTAMP) AS ModifiedDate

FROM adventure_works.bronze.stg_purchase_order_header;


In [0]:
SELECT COUNT(*) AS total_rows
FROM adventure_works.silver.stg_purchase_order_header;

In [0]:
DESCRIBE adventure_works.silver.stg_purchase_order_header;

In [0]:

CREATE TABLE adventure_works.silver.dim_vendor
AS
SELECT
    TRY_CAST(_c0 AS INT) AS BusinessEntityID,
    _c1 AS AccountNumber,
    _c2 AS Name,
    TRY_CAST(_c3 AS TINYINT) AS CreditRating,

    CASE
        WHEN _c4 IN ('1', 'true', 'TRUE') THEN TRUE
        WHEN _c4 IN ('0', 'false', 'FALSE') THEN FALSE
        ELSE NULL
    END AS PreferredVendorStatus,

    CASE
        WHEN _c5 IN ('1', 'true', 'TRUE') THEN TRUE
        WHEN _c5 IN ('0', 'false', 'FALSE') THEN FALSE
        ELSE NULL
    END AS ActiveFlag,

    _c6 AS PurchasingWebServiceURL,
    TRY_CAST(_c7 AS TIMESTAMP) AS ModifiedDate

FROM adventure_works.bronze.dim_vendor;


In [0]:

SELECT COUNT(*) AS total_vendor_rows
FROM adventure_works.silver.dim_vendor;


In [0]:

DESCRIBE adventure_works.silver.dim_vendor;


In [0]:

SELECT *
FROM adventure_works.silver.dim_vendor
LIMIT 10;


In [0]:

CREATE TABLE adventure_works.silver.stg_sales_order_header
AS
SELECT
    TRY_CAST(_c0 AS INT) AS SalesOrderID,
    TRY_CAST(_c1 AS TINYINT) AS RevisionNumber,
    TRY_CAST(_c2 AS TIMESTAMP) AS OrderDate,
    TRY_CAST(_c3 AS TIMESTAMP) AS DueDate,
    TRY_CAST(_c4 AS TIMESTAMP) AS ShipDate,
    TRY_CAST(_c5 AS TINYINT) AS Status,

    CASE
        WHEN _c6 IN ('1', 'true', 'TRUE') THEN TRUE
        WHEN _c6 IN ('0', 'false', 'FALSE') THEN FALSE
        ELSE NULL
    END AS OnlineOrderFlag,

    _c7 AS SalesOrderNumber,
    _c8 AS PurchaseOrderNumber,
    _c9 AS AccountNumber,
    TRY_CAST(_c10 AS INT) AS CustomerID,
    TRY_CAST(_c11 AS INT) AS SalesPersonID,
    TRY_CAST(_c12 AS INT) AS TerritoryID,
    TRY_CAST(_c13 AS INT) AS BillToAddressID,
    TRY_CAST(_c14 AS INT) AS ShipToAddressID,
    TRY_CAST(_c15 AS INT) AS ShipMethodID,
    TRY_CAST(_c16 AS INT) AS CreditCardID,
    _c17 AS CreditCardApprovalCode,
    TRY_CAST(_c18 AS INT) AS CurrencyRateID,
    TRY_CAST(_c19 AS DECIMAL(19,4)) AS SubTotal,
    TRY_CAST(_c20 AS DECIMAL(19,4)) AS TaxAmt,
    TRY_CAST(_c21 AS DECIMAL(19,4)) AS Freight,
    TRY_CAST(_c22 AS DECIMAL(19,4)) AS TotalDue,
    _c23 AS Comment,
    _c24 AS rowguid,
    TRY_CAST(_c25 AS TIMESTAMP) AS ModifiedDate

FROM adventure_works.bronze.stg_sales_order_header;


In [0]:
SELECT COUNT(*) AS total_rows
FROM adventure_works.silver.stg_sales_order_header;

In [0]:
DESCRIBE adventure_works.silver.stg_sales_order_header;

In [0]:

SELECT
    'PurchaseOrderDetail' AS table_name,
    COUNT(*) AS total_rows
FROM adventure_works.silver.stg_purchase_order_detail

UNION ALL

SELECT
    'PurchaseOrderHeader',
    COUNT(*)
FROM adventure_works.silver.stg_purchase_order_header

UNION ALL

SELECT
    'Vendor',
    COUNT(*)
FROM adventure_works.silver.dim_vendor;


##Validating the join keys

In [0]:

SELECT
    PurchaseOrderID,
    COUNT(*) AS duplicate_count
FROM adventure_works.silver.stg_purchase_order_header
GROUP BY PurchaseOrderID
HAVING COUNT(*) > 1;


In [0]:

SELECT COUNT(*) AS unmatched_purchase_details
FROM adventure_works.silver.stg_purchase_order_detail d
LEFT ANTI JOIN adventure_works.silver.stg_purchase_order_header h
    ON d.PurchaseOrderID = h.PurchaseOrderID;


In [0]:
SELECT
    SalesOrderID,
    OrderDate
FROM adventure_works.silver.stg_sales_order_header
LIMIT 10;

##Creating Gold Layer

In [0]:

CREATE SCHEMA IF NOT EXISTS adventure_works.gold;


In [0]:

CREATE TABLE adventure_works.gold.fact_purchases
AS
SELECT
    d.PurchaseOrderID,
    d.PurchaseOrderDetailID,
    d.ProductID,
    h.VendorID,
    h.OrderDate,
    d.DueDate,
    d.OrderQty,
    d.UnitPrice,
    d.LineTotal,
    d.ReceivedQty,
    d.RejectedQty,
    d.StockedQty,
    d.ModifiedDate

FROM adventure_works.silver.stg_purchase_order_detail d

INNER JOIN adventure_works.silver.stg_purchase_order_header h
    ON d.PurchaseOrderID = h.PurchaseOrderID;


In [0]:

SELECT COUNT(*) AS total_purchase_rows
FROM adventure_works.gold.fact_purchases;


In [0]:

SELECT
    PurchaseOrderDetailID,
    COUNT(*) AS duplicate_count
FROM adventure_works.gold.fact_purchases
GROUP BY PurchaseOrderDetailID
HAVING COUNT(*) > 1;


In [0]:

SELECT *
FROM adventure_works.gold.fact_purchases
LIMIT 10;


## Query 1: Find the 5 products with the thinnest margins

In [0]:

WITH sales_prices AS (
    SELECT
        ProductID,
        SUM(LineTotal) / SUM(OrderQty) AS selling_price_per_unit
    FROM adventure_works.default.fact_sales
    GROUP BY ProductID
),

purchase_prices AS (
    SELECT
        ProductID,
        SUM(LineTotal) / SUM(OrderQty) AS purchase_price_per_unit
    FROM adventure_works.gold.fact_purchases
    GROUP BY ProductID
)

SELECT
    p.ProductID,
    p.Name AS product_name,

    ROUND(s.selling_price_per_unit, 2)
        AS selling_price,

    ROUND(b.purchase_price_per_unit, 2)
        AS purchase_price,

    ROUND(
        s.selling_price_per_unit - b.purchase_price_per_unit,
        2
    ) AS margin_per_unit

FROM sales_prices s

INNER JOIN purchase_prices b
    ON s.ProductID = b.ProductID

INNER JOIN adventure_works.default.dim_product p
    ON s.ProductID = p.ProductID

ORDER BY margin_per_unit ASC

LIMIT 5;


## Query 2: Monthly units sold vs. units purchased

In [0]:

DESCRIBE adventure_works.default.fact_sales;


In [0]:
SELECT *
FROM adventure_works.bronze.sales_order_detail
;

In [0]:

WITH monthly_sales AS (
    SELECT
        f.ProductID,
        DATE_TRUNC('MONTH', h.OrderDate) AS sales_month,
        SUM(f.OrderQty) AS units_sold
    FROM adventure_works.default.fact_sales f
    INNER JOIN adventure_works.silver.stg_sales_order_header h
        ON f.SalesOrderID = h.SalesOrderID
    GROUP BY
        f.ProductID,
        DATE_TRUNC('MONTH', h.OrderDate)
),

monthly_purchases AS (
    SELECT
        ProductID,
        DATE_TRUNC('MONTH', OrderDate) AS purchase_month,
        SUM(OrderQty) AS units_purchased
    FROM adventure_works.gold.fact_purchases
    GROUP BY
        ProductID,
        DATE_TRUNC('MONTH', OrderDate)
)

SELECT
    COALESCE(s.ProductID, p.ProductID) AS ProductID,
    COALESCE(s.sales_month, p.purchase_month) AS month,
    COALESCE(s.units_sold, 0) AS units_sold,
    COALESCE(p.units_purchased, 0) AS units_purchased,

    COALESCE(s.units_sold, 0)
        - COALESCE(p.units_purchased, 0) AS unit_difference,

    CASE
        WHEN COALESCE(s.units_sold, 0) >
             COALESCE(p.units_purchased, 0)
            THEN 'Selling faster than restocking'

        WHEN COALESCE(s.units_sold, 0) <
             COALESCE(p.units_purchased, 0)
            THEN 'Restocking faster than selling'

        ELSE 'Balanced'
    END AS stock_status

FROM monthly_sales s
FULL OUTER JOIN monthly_purchases p
    ON s.ProductID = p.ProductID
    AND s.sales_month = p.purchase_month

ORDER BY month, ProductID;


In [0]:

WITH monthly_sales AS (
    SELECT
        f.ProductID,
        DATE_TRUNC('MONTH', h.OrderDate) AS month,
        SUM(f.OrderQty) AS units_sold
    FROM adventure_works.default.fact_sales f
    JOIN adventure_works.silver.stg_sales_order_header h
        ON f.SalesOrderID = h.SalesOrderID
    GROUP BY f.ProductID, DATE_TRUNC('MONTH', h.OrderDate)
),

monthly_purchases AS (
    SELECT
        ProductID,
        DATE_TRUNC('MONTH', OrderDate) AS month,
        SUM(OrderQty) AS units_purchased
    FROM adventure_works.gold.fact_purchases
    GROUP BY ProductID, DATE_TRUNC('MONTH', OrderDate)
),

monthly_comparison AS (
    SELECT
        COALESCE(s.ProductID, p.ProductID) AS ProductID,
        COALESCE(s.month, p.month) AS month,
        COALESCE(s.units_sold, 0) AS units_sold,
        COALESCE(p.units_purchased, 0) AS units_purchased
    FROM monthly_sales s
    FULL OUTER JOIN monthly_purchases p
        ON s.ProductID = p.ProductID
        AND s.month = p.month
)

SELECT
    m.ProductID,
    d.Name AS product_name,
    SUM(m.units_sold) AS total_units_sold,
    SUM(m.units_purchased) AS total_units_purchased,
    SUM(m.units_sold - m.units_purchased) AS net_difference,

    SUM(
        CASE WHEN m.units_sold > m.units_purchased
        THEN 1 ELSE 0 END
    ) AS months_selling_faster

FROM monthly_comparison m
JOIN adventure_works.default.dim_product d
    ON m.ProductID = d.ProductID

GROUP BY m.ProductID, d.Name

HAVING SUM(m.units_sold) > SUM(m.units_purchased)

ORDER BY net_difference DESC

LIMIT 10;
